In [52]:
import os
import json
import numpy as np
import torch

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer
)

from tqdm.auto import tqdm
tqdm.pandas()

import transformers
transformers.logging.set_verbosity_error()


In [53]:
DATA_PATH = "/kaggle/input/auto-tagger-dataset/autotagger_ner_500.jsonl"

dataset = load_dataset("json", data_files={"data": DATA_PATH})["data"]

# 70% train, 30% temp
dataset = dataset.train_test_split(test_size=0.30, seed=42)

# Split that 30% into 15% validation + 15% test
temp = dataset["test"].train_test_split(test_size=0.50, seed=42)

train_dataset = dataset["train"]
valid_dataset = temp["train"]
test_dataset  = temp["test"]

print("Train:", len(train_dataset))
print("Valid:", len(valid_dataset))
print("Test:", len(test_dataset))

Train: 350
Valid: 75
Test: 75


In [54]:
label_list = [
    "O",
    "B-MAKE",
    "B-MODEL",
    "B-YEAR",
    "B-PART",
    "I-PART",
    "B-BRAND"
]

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [55]:
def tokenize_and_align_labels(example):
    tokenized = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    all_labels = []
    word_ids = tokenized.word_ids()
    prev_word = None

    for idx, word_idx in enumerate(word_ids):
        if word_idx is None:
            all_labels.append(-100)
        else:
            label = example["labels"][word_idx]

            if word_idx != prev_word:
                all_labels.append(label2id[label])
            else:
                if label.startswith("B-"):
                    label = "I-" + label[2:]
                all_labels.append(label2id.get(label, label2id["O"]))

        prev_word = word_idx

    tokenized["labels"] = all_labels
    return tokenized


encoded_train = train_dataset.map(tokenize_and_align_labels)
encoded_valid = valid_dataset.map(tokenize_and_align_labels)
encoded_test  = test_dataset.map(tokenize_and_align_labels)

print("Tokenization complete.")


Map:   0%|          | 0/75 [00:00<?, ? examples/s]

Tokenization complete.


In [56]:
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)


In [57]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    correct, total = 0, 0

    for pred_seq, label_seq in zip(preds, labels):
        for p, l in zip(pred_seq, label_seq):
            if l == -100:
                continue
            total += 1
            if p == l:
                correct += 1

    return {"accuracy": correct / total if total > 0 else 0}

In [66]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="/kaggle/working/autotagger-ner-model",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=6,
    weight_decay=0.01,
    logging_steps=20,
    disable_tqdm=False,     # progress bar
    report_to=[]            # disable wandb
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train,
    eval_dataset=encoded_valid,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

# Explicit save
trainer.save_model("/kaggle/working/autotagger-ner-model")
tokenizer.save_pretrained("/kaggle/working/autotagger-ner-model")


/tmp/ipykernel_47/729936156.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
20,0.000600
40,0.001700
60,0.000300
80,0.000200
100,0.000100
120,0.000100
140,0.000100
160,0.000100
180,0.000100
200,0.000100


('/kaggle/working/autotagger-ner-model/tokenizer_config.json',
 '/kaggle/working/autotagger-ner-model/special_tokens_map.json',
 '/kaggle/working/autotagger-ner-model/vocab.txt',
 '/kaggle/working/autotagger-ner-model/added_tokens.json',
 '/kaggle/working/autotagger-ner-model/tokenizer.json')

In [ ]:
trainer.save_model("/kaggle/working/autotagger-ner-model")
tokenizer.save_pretrained("/kaggle/working/autotagger-ner-model")


In [59]:
results = trainer.evaluate(encoded_test)
results

{'eval_loss': 0.0007303431048057973,
 'eval_accuracy': 1.0,
 'eval_runtime': 0.0774,
 'eval_samples_per_second': 968.608,
 'eval_steps_per_second': 129.148,
 'epoch': 6.0}

In [63]:
def extract_entities(text):
    tokens = text.split()

    # Tokenize for model input
    enc = tokenizer(tokens, is_split_into_words=True, return_tensors="pt")
    enc = {k: v.to(model.device) for k, v in enc.items()}

    with torch.no_grad():
        outputs = model(**enc)
        preds = outputs.logits.argmax(dim=-1)[0].tolist()

    # Tokenize again (no tensors) just to get word_ids
    raw = tokenizer(tokens, is_split_into_words=True)
    word_ids = raw.word_ids()

    entity_map = {"MAKE": [], "MODEL": [], "YEAR": [], "PART": [], "BRAND": []}
    curr_tokens = []
    curr_type = None
    prev_word_id = None

    for pred_id, word_id in zip(preds, word_ids):
        # Skip special tokens and repeated subwords
        if word_id is None or word_id == prev_word_id:
            continue
        prev_word_id = word_id

        label = id2label[pred_id]
        word = tokens[word_id]

        if label.startswith("B-"):
            # close previous span
            if curr_tokens:
                entity_map[curr_type].append(" ".join(curr_tokens))
            curr_type = label.split("-")[1]
            curr_tokens = [word]

        elif label.startswith("I-"):
            typ = label.split("-")[1]
            if curr_type == typ:
                curr_tokens.append(word)
            else:
                # inconsistent I- tag; start a new span
                if curr_tokens:
                    entity_map[curr_type].append(" ".join(curr_tokens))
                curr_type = typ
                curr_tokens = [word]

        else:  # "O"
            if curr_tokens:
                entity_map[curr_type].append(" ".join(curr_tokens))
            curr_tokens = []
            curr_type = None

    # Close last span
    if curr_tokens:
        entity_map[curr_type].append(" ".join(curr_tokens))

    # Simplify: take first instance of each entity type
    return {k: (v[0] if v else None) for k, v in entity_map.items()}


In [65]:
extract_entities("2015 Dodge Ram front strut OEM")


{'MAKE': 'Dodge',
 'MODEL': 'Ram',
 'YEAR': '2015',
 'PART': 'front strut',
 'BRAND': 'OEM'}